# LLM 軽量化

LLM の軽量化は、品質、メモリ、レイテンシ、コストの制約を同時に満たす設計である。重みを小さくするだけでは足りない。長文生成では KV cache が増え、運用では batching や decoding の流し方も効く。

量子化、pruning、蒸留、LoRA、GQA、continuous batching、speculative decoding は、それぞれ削る対象が違う。まず何が詰まっているかを数字で切り分ける。

In [ ]:
import math
import random


def gib(num_bytes):
    return num_bytes / (1024 ** 3)


def param_bytes(params, bits):
    return params * bits / 8

models = {'7B': 7_000_000_000, '13B': 13_000_000_000, '70B': 70_000_000_000}
for name, params in models.items():
    print(name, 'FP16 GiB=', round(gib(param_bytes(params, 16)), 2), 'INT4 GiB=', round(gib(param_bytes(params, 4)), 2))

重みメモリはパラメータ数と bit 幅に比例する。FP16 から INT4 へ落とすと、理論上は重み部分が 4 分の 1 になる。

In [ ]:
def kv_cache_bytes(batch, seq_len, layers, kv_heads, head_dim, bits):
    elements = batch * seq_len * layers * kv_heads * head_dim * 2
    return elements * bits / 8

cfg = {'batch': 4, 'layers': 32, 'kv_heads': 32, 'head_dim': 128}
for seq_len in [1024, 4096, 8192, 32768]:
    fp16 = kv_cache_bytes(seq_len=seq_len, bits=16, **cfg)
    int8 = kv_cache_bytes(seq_len=seq_len, bits=8, **cfg)
    print('seq=', seq_len, 'KV FP16 GiB=', round(gib(fp16), 2), 'KV INT8 GiB=', round(gib(int8), 2))

KV cache は batch、文脈長、層数、KV head 数、head 次元に比例する。長文になるほど、重みより KV cache が先に詰まることがある。

In [ ]:
def quantize_uniform(values, bits):
    lo = min(values)
    hi = max(values)
    levels = (1 << bits) - 1
    if hi == lo:
        return values[:], 0.0
    scale = (hi - lo) / levels
    q = [round((v - lo) / scale) for v in values]
    deq = [lo + x * scale for x in q]
    mse = sum((a - b) ** 2 for a, b in zip(values, deq)) / len(values)
    return deq, mse

random.seed(5)
weights = [random.gauss(0.0, 0.6) for _ in range(2048)]
for bits in [8, 6, 4, 3]:
    _, mse = quantize_uniform(weights, bits)
    print(bits, 'bit MSE=', round(mse, 6), 'bytes=', round(len(weights) * bits / 8))

量子化は、値を少ない bit の段階へ丸める。bit を下げるほどメモリは減るが、丸め誤差は増える。品質評価なしに bit 幅だけ決めると壊れやすい。

In [ ]:
rows, cols = 64, 64
matrix = [[random.uniform(-1.0, 1.0) for _ in range(cols)] for _ in range(rows)]
flat = sorted(abs(x) for row in matrix for x in row)
threshold = flat[len(flat) // 2]
unstructured_nonzero = sum(abs(x) >= threshold for row in matrix for x in row)

row_norms = [(sum(abs(x) for x in row), i) for i, row in enumerate(matrix)]
kept_rows = {i for _, i in sorted(row_norms, reverse=True)[:rows // 2]}
structured_shape = (len(kept_rows), cols)

print('unstructured nonzero ratio:', round(unstructured_nonzero / (rows * cols), 3))
print('structured new shape:', structured_shape)
print('dense matmul work:', rows * cols)
print('structured matmul work:', structured_shape[0] * structured_shape[1])

Unstructured pruning は細かい重みを 0 にする。Structured pruning は行や列ごと削る。実行速度に効きやすいのは、行列サイズそのものが小さくなる structured 側である。

In [ ]:
def softmax(logits, temperature=1.0):
    scaled = [v / temperature for v in logits]
    m = max(scaled)
    exps = [math.exp(v - m) for v in scaled]
    total = sum(exps)
    return [v / total for v in exps]


def kl(p, q):
    return sum(pi * math.log(pi / max(qi, 1e-12)) for pi, qi in zip(p, q) if pi > 0)

teacher = [6.0, 3.0, 1.0, -1.0]
student = [4.5, 2.0, 0.4, -0.5]
for temp in [1.0, 2.0, 4.0]:
    p_t = softmax(teacher, temp)
    p_s = softmax(student, temp)
    print('T=', temp, 'teacher=', [round(v, 3) for v in p_t], 'KL=', round(kl(p_t, p_s), 4))

蒸留では、正解ラベルだけでなく教師モデルの確率分布を学生モデルへ移す。温度を上げると、教師がどの候補を少し近いと見ているかが見えやすくなる。

In [ ]:
def lora_params(hidden, rank, layers, projections_per_layer=4):
    return layers * projections_per_layer * (hidden * rank + rank * hidden)

hidden = 4096
layers = 32
full_trainable = layers * 4 * hidden * hidden
for rank in [4, 8, 16, 32]:
    trainable = lora_params(hidden, rank, layers)
    print('rank', rank, 'LoRA params=', trainable, 'ratio=', round(trainable / full_trainable, 5))

base_7b_int4 = param_bytes(7_000_000_000, 4)
print('7B INT4 base GiB:', round(gib(base_7b_int4), 2))

LoRA は学習対象を低ランク行列に限定する。軽くなる主対象は追加学習であり、QLoRA はベース重みも 4bit などで保持して学習メモリをさらに下げる。

In [ ]:
def attention_work(seq_len):
    return seq_len * seq_len


def kv_heads_after_gqa(query_heads, group_size):
    return max(1, query_heads // group_size)

for seq in [1024, 4096, 16384]:
    print('seq', seq, 'attention O(L^2)=', attention_work(seq))

for group in [1, 2, 4, 8, 32]:
    kv_heads = kv_heads_after_gqa(32, group)
    kv = kv_cache_bytes(batch=4, seq_len=8192, layers=32, kv_heads=kv_heads, head_dim=128, bits=16)
    print('GQA group', group, 'kv_heads=', kv_heads, 'KV GiB=', round(gib(kv), 2))

Attention の主要計算は文脈長の二乗で増える。GQA/MQA は query head に対して KV head を共有し、KV cache を減らす。共有を増やすほどメモリは減るが、品質確認が必要になる。

In [ ]:
def static_batch_steps(lengths, slots):
    total_steps = 0
    useful_tokens = sum(lengths)
    for i in range(0, len(lengths), slots):
        batch = lengths[i:i + slots]
        total_steps += max(batch) * len(batch)
    return total_steps, useful_tokens / total_steps

lengths = [120, 900, 180, 60, 760, 240, 80, 640]
steps, util = static_batch_steps(lengths, slots=4)
print('static padded steps:', steps, 'utilization:', round(util, 3))
print('token work without padding:', sum(lengths))

Static batching は、短いリクエストも長いリクエストに合わせて待つため無駄が出る。Continuous batching は完了した枠へ次のリクエストを入れ、GPU の空きを減らす。

In [ ]:
def speculative_speedup(draft_block, accept_rate, draft_cost_ratio):
    verified_tokens = max(1e-9, draft_block * accept_rate)
    cost = 1.0 + draft_block * draft_cost_ratio
    baseline_cost = verified_tokens
    return baseline_cost / cost

for accept in [0.35, 0.55, 0.75, 0.9]:
    print('accept', accept, 'speedup=', round(speculative_speedup(6, accept, 0.12), 3))

Speculative decoding は小さい下書きモデルに複数トークンを先に出させ、本体モデルがまとめて検証する。下書きが当たるほど速くなり、外れるほど効果は下がる。

In [ ]:
def recommend(vram_gib, seq_len, needs_training, latency_sensitive):
    plan = []
    weight_7b_int4 = gib(param_bytes(7_000_000_000, 4))
    kv = gib(kv_cache_bytes(batch=4, seq_len=seq_len, layers=32, kv_heads=32, head_dim=128, bits=16))
    if weight_7b_int4 + kv > vram_gib:
        plan.append('INT4 weights')
        plan.append('KV cache INT8 or GQA')
    else:
        plan.append('FP16 or INT8 weights can fit')
    if needs_training:
        plan.append('LoRA or QLoRA')
    if latency_sensitive:
        plan.append('continuous batching')
        plan.append('speculative decoding after acceptance test')
    return plan

for case in [(16, 4096, True, True), (48, 8192, False, True), (24, 32768, True, False)]:
    print(case, '=>', recommend(*case))

軽量化では、まず重み、KV cache、attention 計算、batching のどこが詰まるかを測る。量子化は表現 bit、pruning は構造、蒸留はモデルサイズ、LoRA は学習対象、GQA は KV cache、batching と speculative decoding は推論の流し方に効く。